In [ ]:
import pandas as pd
import joblib
import numpy as np
from pathlib import Path

import warnings
warnings.filterwarnings("ignore")

BASE_DIR = Path.cwd().parent
DATABASE_DIR = BASE_DIR / "app" / "data" / "cleaned" / "data_sales_cleaned.parquet"
DATABASE_SQL = BASE_DIR / "app" / "data" / "cleaned" / "sql_supermarket.parquet"
MODELS_DIR = BASE_DIR / "Models"

In [ ]:
df_sql = pd.read_parquet(DATABASE_SQL)
df_sql

## Load database

In [40]:
pd.set_option('display.max_columns', None)
df = pd.read_parquet(DATABASE_DIR)
df

,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year,unit_price,profit_margin
0,AG-2011-2040,2011-01-01,2011-06-01,Standard Class,Toby Braunhardt,Consumer,Constantine,Algeria,Africa,Africa,OFF-TEN-10000025,Office Supplies,Storage,"Tenex Lockers, Blue",4080000.0,2,0.0,1061400.0,354600.0,Medium,2011,2040000.0,0.26
1,IN-2011-47883,2011-01-01,2011-08-01,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,OFF-SU-10000618,Office Supplies,Supplies,"Acme Trimmer, High Speed",1200000.0,3,0.1,360360.0,97200.0,Medium,2011,400000.0,0.30
2,HU-2011-1220,2011-01-01,2011-05-01,Second Class,Annie Thurman,Consumer,Budapest,Hungary,EMEA,EMEA,OFF-TEN-10001585,Office Supplies,Storage,"Tenex Box, Single Width",660000.0,4,0.0,296400.0,81700.0,High,2011,165000.0,0.45
3,IT-2011-3647632,2011-01-01,2011-05-01,Second Class,Eugene Moren,Home Office,Stockholm,Sweden,EU,North,OFF-PA-10001492,Office Supplies,Paper,"Enermax Note Cards, Premium",450000.0,3,0.5,-260550.0,48200.0,High,2011,150000.0,-0.58
4,CA-2011-1510,2011-02-01,2011-06-01,Standard Class,Magdelene Morse,Consumer,Ontario,Canada,Canada,Canada,TEC-OKI-10002750,Technology,Machines,"Okidata Inkjet, Wireless",3140000.0,1,0.0,31200.0,241000.0,Medium,2011,3140000.0,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25030,CA-2014-115427,2014-12-12,2015-04-01,Standard Class,Erica Bern,Corporate,California,United States,US,West,OFF-BI-10004632,Office Supplies,Binders,GBC Binding covers,210000.0,2,0.2,64750.0,20600.0,Medium,2014,105000.0,0.31
25031,UP-2014-4410,2014-12-12,2015-04-01,Standard Class,Guy Thornton,Consumer,Zaporizhzhya,Ukraine,EMEA,EMEA,OFF-AVE-10003558,Office Supplies,Labels,"Avery Round Labels, Alphabetical",280000.0,4,0.0,61200.0,17000.0,Medium,2014,70000.0,0.22
25032,MX-2014-108574,2014-12-12,2015-04-01,Standard Class,Julia Barnett,Home Office,Tamaulipas,Mexico,LATAM,North,OFF-LA-10004969,Office Supplies,Labels,"Novimex Legal Exhibit Labels, Adjustable",170000.0,3,0.0,6600.0,13200.0,Medium,2014,56700.0,0.04
25033,MO-2014-2560,2014-12-12,2015-05-01,Standard Class,Liz Preis,Consumer,Souss-Massa-Draâ,Morocco,Africa,Africa,OFF-WIL-10001069,Office Supplies,Binders,"Wilson Jones Hole Reinforcements, Clear",40000.0,1,0.0,4200.0,4900.0,Medium,2014,40000.0,0.10


## Feature engineering

In [41]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()

df_model = df.copy()
df_model = df_model.drop(columns=["order_id", "order_date", "ship_date", "product_id"], errors="ignore")

object_cols = df_model.select_dtypes(include=["object"]).columns

for col in object_cols:
    df_model[col] = label_encoder.fit_transform(df_model[col])

df_model

,ship_mode,customer_name,segment,state,country,market,region,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year,unit_price,profit_margin
0,3,751,0,255,2,1,0,1,14,3266,4080000.0,2,0.0,1061400.0,354600.0,3,2011,2040000.0,0.26
1,3,398,0,701,6,0,9,1,15,154,1200000.0,3,0.1,360360.0,97200.0,3,2011,400000.0,0.30
2,2,48,0,175,56,3,5,1,14,3235,660000.0,4,0.0,296400.0,81700.0,1,2011,165000.0,0.45
3,2,275,2,937,123,4,7,1,12,1253,450000.0,3,0.5,-260550.0,48200.0,1,2011,150000.0,-0.58
4,3,483,0,746,22,2,1,2,11,2541,3140000.0,1,0.0,31200.0,241000.0,3,2011,3140000.0,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25030,3,264,1,192,139,6,12,1,3,1405,210000.0,2,0.2,64750.0,20600.0,3,2014,105000.0,0.31
25031,3,316,0,1080,136,3,5,1,10,430,280000.0,4,0.0,61200.0,17000.0,3,2014,70000.0,0.22
25032,3,402,2,962,81,5,7,1,10,2416,170000.0,3,0.0,6600.0,13200.0,3,2014,56700.0,0.04
25033,3,472,0,921,85,1,0,1,3,3383,40000.0,1,0.0,4200.0,4900.0,3,2014,40000.0,0.10


## Split data into X and y

In [42]:
X = df_model.drop(columns=["sales"], errors="ignore")
y = df_model["sales"]
X

,ship_mode,customer_name,segment,state,country,market,region,category,sub_category,product_name,quantity,discount,profit,shipping_cost,order_priority,year,unit_price,profit_margin
0,3,751,0,255,2,1,0,1,14,3266,2,0.0,1061400.0,354600.0,3,2011,2040000.0,0.26
1,3,398,0,701,6,0,9,1,15,154,3,0.1,360360.0,97200.0,3,2011,400000.0,0.30
2,2,48,0,175,56,3,5,1,14,3235,4,0.0,296400.0,81700.0,1,2011,165000.0,0.45
3,2,275,2,937,123,4,7,1,12,1253,3,0.5,-260550.0,48200.0,1,2011,150000.0,-0.58
4,3,483,0,746,22,2,1,2,11,2541,1,0.0,31200.0,241000.0,3,2011,3140000.0,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25030,3,264,1,192,139,6,12,1,3,1405,2,0.2,64750.0,20600.0,3,2014,105000.0,0.31
25031,3,316,0,1080,136,3,5,1,10,430,4,0.0,61200.0,17000.0,3,2014,70000.0,0.22
25032,3,402,2,962,81,5,7,1,10,2416,3,0.0,6600.0,13200.0,3,2014,56700.0,0.04
25033,3,472,0,921,85,1,0,1,3,3383,1,0.0,4200.0,4900.0,3,2014,40000.0,0.10


## Load models

In [43]:
# Load pickle files in MODELS_DIR
def load_models(models_dir: Path):
    models = {}

    for pkl_path in models_dir.glob("*.pkl"):
        if "fraud_ml_models" in pkl_path.parts:
            continue  # Skip the "fraud_ml_models" directory

        model_name = pkl_path.stem.replace("model_", "")
        models[model_name] = joblib.load(pkl_path)

    return models

sales_models = load_models(MODELS_DIR)
print(f"Load models: {sales_models.keys()}")

Load models: dict_keys(['RandomForestRegressor', 'LinearRegression', 'XGBRegressor'])


## Get model features

In [44]:
def get_model_features(model):
    if hasattr(model, "feature_names_in_"):
        return list(model.feature_names_in_)
    if hasattr(model, "get_booster"):
        try:
            return model.get_booster().feature_names
        except Exception:
            return None        
    if hasattr(model, "feature_importances_"):
        return None  # Feature importances are available, but feature names are not directly accessible
    
    return None 

# Compare model with X features
def compare_model_with_X(model, X):
    model_features = get_model_features(model)
    x_features = list(X.columns)

    print(f"Model: {type(model).__name__}")
    print(f"X Shape: {X.shape}")

    if model_features is None:
        print("Model feature names are not available.")
        return
    
    print(f"Expected features ({len(model_features)}): {model_features}")
    print(f"X features ({len(x_features)}): {x_features}")

    missing_in_X = [f for f in model_features if f not in x_features]
    extra_in_X = [f for f in x_features if f not in model_features]

    print(f"Missing in X: {missing_in_X}")
    print(f"Extra in X: {extra_in_X}")

In [45]:
for model_name, model in sales_models.items():
    compare_model_with_X(model, X)
    print("-" * 50)

Model: RandomForestRegressor
X Shape: (25035, 18)
Model feature names are not available.
--------------------------------------------------
Model: LinearRegression
X Shape: (25035, 18)
Model feature names are not available.
--------------------------------------------------
Model: RandomForestRegressor
X Shape: (25035, 18)
Model feature names are not available.
--------------------------------------------------


In [46]:
def show_feature_importance(model, feature_names):
    if hasattr(model, "feature_importances_"):
        importance_df = pd.DataFrame({
            "feature": feature_names,
            "importance": model.feature_importances_
        }).sort_values("importance", ascending=False)
        return importance_df
    
    return None

fi = show_feature_importance(sales_models["XGBRegressor"], X.columns)
fi

,feature,importance
13,shipping_cost,0.612058
16,unit_price,0.227411
10,quantity,0.109055
12,profit,0.015791
14,order_priority,0.005950
17,profit_margin,0.005365
9,product_name,0.003790
1,customer_name,0.003687
3,state,0.003229
0,ship_mode,0.002452
